# 03 — Gold: bid performance

Agregados voltados pro negócio. Duas regras moldam esta camada:

**Toda taxa é reportada duas vezes** — incluindo e excluindo registros
carregados em lote. Um único número aqui seria enganoso, e qual dos dois é
"correto" depende da pergunta sendo feita.

**A taxa de vitória é reportada por contagem e por valor.** As duas diferem
em cerca de cinco pontos, e só a taxa ponderada por valor reflete a
realidade comercial.

A duração do ciclo de venda está deliberadamente ausente. Mais da metade
dos timestamps de fechamento foi gravada em sessões de encerramento em
massa, segundos de intervalo, meses depois do fato — qualquer duração
derivada deles seria ficção.

**`clients_clean` é uma dimensão SCD Type 2** — um cliente pode ter mais de
uma linha. O join abaixo associa cada proposta à versão do cliente *que
estava vigente quando a proposta foi feita* (`created_at` entre
`valid_from` e `valid_to`), não simplesmente à linha que calha de ser
atual hoje. Um join simples por `client_id` faria fan-out silencioso de
toda proposta de um cliente renovado em duas linhas, contando em dobro.

Cerca de 6% das propostas não vão casar com nenhuma versão (a checagem de
integridade referencial do Silver conta isso com precisão) — essas linhas
mantêm seu outcome e valor mas ficam com segment/state/executive NULL, e
aparecem como um bucket `None` em toda tabela segmentada abaixo, em vez de
serem descartadas ou forçadas pro período errado.

In [0]:
from pyspark.sql import functions as F
import sys
sys.path.append("/Workspace/Users/samuelsouzadias@outlook.com/bid-win-loss-analytics/en")
from transforms import point_in_time_join

spark.sql("CREATE SCHEMA IF NOT EXISTS gold.bid")

bids = spark.table("silver.bid.bids_clean")
clients = spark.table("silver.bid.clients_clean")

# Join point-in-time: os atributos do cliente como eram quando a proposta
# foi feita, não como são hoje. Lista explícita de colunas no select — com
# client_id presente nos dois lados da condição de join, `b.*` mais colunas
# `c.*` nomeadas evita um "client_id" ambíguo no resultado.
fact = point_in_time_join(bids, clients).filter(F.col("outcome").isNotNull())      # somente propostas fechadas
# Propostas que não casaram com nenhuma versão de cliente — o Silver já
# checa isso (ver a célula "cobertura point-in-time" do 02), reverificado
# aqui já que é a premissa da qual todo este join depende.
unmatched = fact.filter(F.col("client_sk").isNull()).count()
if unmatched:
    print(f"AVISO — {unmatched} propostas fechadas não casaram com nenhuma versão de cliente")

## Métricas principais

`win_rate_by_count` trata uma proposta de R$20 mil e uma de R$2 milhões
como idênticas. `win_rate_by_value` não. Publicar só a primeira superestima
o desempenho.

In [0]:
def performance(df, *dims):
    return (
        df.groupBy(*dims)
          .agg(
              F.count("*").alias("bids_closed"),
              F.sum("outcome").alias("bids_won"),
              F.round(F.avg("outcome") * 100, 1).alias("win_rate_by_count"),
              F.round(
                  F.sum(F.when(F.col("outcome") == 1, F.col("contract_value_brl")).otherwise(0))
                  / F.sum("contract_value_brl") * 100, 1
              ).alias("win_rate_by_value"),
              F.round(F.sum("contract_value_brl"), 2).alias("value_bid_brl"),
          )
    )


overall = performance(fact, F.lit(True).alias("_all")).drop("_all")
by_channel = performance(fact, "is_bulk_load")

overall.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_overall")
by_channel.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_by_channel")

display(by_channel)

## Segmentação, com o confound isolado

Cada dimensão é reportada em todas as propostas fechadas, e de novo somente
nas orgânicas. O gap entre as duas colunas é o tamanho do artefato de
migração — e, para pelo menos um executivo, é a diferença entre "abaixo da
média" e "acima da média".

In [0]:
organic = fact.filter(~F.col("is_bulk_load"))

for dim in ["segment", "state", "account_executive"]:
    combined = (
        performance(fact, dim)
        .select(dim, "bids_closed", F.col("win_rate_by_count").alias("wr_all"))
        .join(
            performance(organic, dim)
              .select(dim,
                      F.col("bids_closed").alias("bids_organic"),
                      F.col("win_rate_by_count").alias("wr_organic")),
            dim, "left",
        )
        .withColumn("artefact_gap", F.round(F.col("wr_organic") - F.col("wr_all"), 1))
        .orderBy(F.col("bids_closed").desc())
    )
    combined.write.format("delta").mode("overwrite").saveAsTable(f"gold.bid.performance_by_{dim}")
    display(combined)

## Efeito do valor do contrato

Os quartis são calculados sobre as propostas fechadas. Se a taxa de vitória
cai conforme o valor sobe, o relatório por contagem está sendo
sistematicamente favorável demais.

Os cortes de quartil vêm de `approxQuantile`, não de `ntile()` sobre uma
window sem partição. `ntile` precisa que toda linha seja ordenada num único
nó para atribuir ranks exatos — inofensivo com 1.600 linhas, mas é o padrão
por trás do aviso `WindowExpression: No Partition Defined` que o Spark
lança, e para de escalar bem antes do resto deste pipeline. `approxQuantile`
encontra os três pontos de corte com um algoritmo distribuído baseado em
sketch, e então classifica cada linha com um `when/otherwise` simples — sem
nenhuma etapa de shuffle-para-um-nó.

In [0]:
# Cortes de quartil aproximados (relativeError=0.01 — precisão de sobra
# para bandas de relatório, e barato independente do tamanho da tabela).
q1, q2, q3 = fact.approxQuantile("contract_value_brl", [0.25, 0.5, 0.75], 0.01)

value_bands = fact.withColumn(
    "value_quartile",
    F.when(F.col("contract_value_brl") <= q1, 1)
     .when(F.col("contract_value_brl") <= q2, 2)
     .when(F.col("contract_value_brl") <= q3, 3)
     .otherwise(4),
)

by_value = (
    performance(value_bands, "value_quartile")
    .orderBy("value_quartile")
)

by_value.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_by_value_band")
display(by_value)

## Motivos de perda, e o quanto pouco eles cobrem do quadro

O número de cobertura é o ponto desta tabela. Uma distribuição de motivos
construída sobre uma fração de um dígito das perdas é um sinal, não uma
estimativa populacional, e os dois precisam ser publicados juntos.

`share_pct` costumava dividir por uma soma calculada com
`Window.partitionBy()` — uma especificação de partição vazia, o mesmo
problema dos quartis acima: força o DataFrame inteiro pra um único nó só
pra obter um total. Com apenas um punhado de motivos distintos, o total é
um número — calculado uma vez com `.collect()` e reutilizado como um
escalar simples.

In [0]:
losses = bids.filter(F.col("outcome") == 0)

coverage = losses.agg(
    F.count("*").alias("losses_total"),
    F.sum(F.col("has_loss_reason").cast("int")).alias("losses_with_reason"),
    F.round(F.avg(F.col("has_loss_reason").cast("int")) * 100, 1).alias("coverage_pct"),
    F.sum(F.col("competitor_is_placeholder").cast("int")).alias("placeholder_attributed"),
)

reasons_raw = (
    losses.filter(F.col("has_loss_reason"))
          .groupBy("loss_reason")
          .agg(F.count("*").alias("losses"))
)

# Um número, calculado uma vez — mais barato e mais claro que uma soma via window.
total_with_reason = reasons_raw.agg(F.sum("losses")).collect()[0][0]

reasons = (
    reasons_raw
    .withColumn("share_pct", F.round(F.col("losses") / F.lit(total_with_reason) * 100, 1))
    .orderBy(F.col("losses").desc())
)

coverage.write.format("delta").mode("overwrite").saveAsTable("gold.bid.loss_reason_coverage")
reasons.write.format("delta").mode("overwrite").saveAsTable("gold.bid.loss_reasons")

display(coverage)
display(reasons)

## Pipeline em aberto

Propostas com outcome NULL. Reportadas separadamente para que nunca entrem
silenciosamente na coluna de perdas, o que subestimaria a taxa de vitória
em cerca de um terço.

Mesmo join point-in-time do `fact` acima — um join simples por `client_id`
contra a tabela `clients` (SCD Type 2) faria fan-out de toda proposta
aberta de um cliente renovado, contando em dobro aqui também.

In [0]:
open_bids = bids.filter(F.col("outcome").isNull())
pipeline = point_in_time_join(open_bids, clients).groupBy("segment").agg(F.count("*").alias("bids_open"), F.round(F.sum("contract_value_brl"), 2).alias("value_open_brl")).orderBy(F.col("value_open_brl").desc())

pipeline.write.format("delta").mode("overwrite").saveAsTable("gold.bid.open_pipeline")
display(pipeline)